# LADD Multimodal Freshness & Formalin-Adulteration Detection — Kaggle training
Built from the project methodology. This notebook:
1. Downloads the LADD dataset directly from Mendeley Data (DOI 10.17632/6n9zbrp75r.1) with SHA-256 verification
2. Trains the hybrid MobileNetV2 + telemetry MLP fusion model (freshness index 0-100)
3. Runs the Mahalanobis 'Chemical Masking' anti-adulteration module
4. Exports an int8 TFLite artifact for ESP32-S3
5. Generates all 6 thesis figures (300 DPI, PNG + PDF)

**Setup:** Settings -> Accelerator -> **GPU T4 x2** or P100. Settings -> Internet -> **ON**. NOTE: Kaggle notebook Settings -> Internet must be ON (requires phone-verified Kaggle account).


In [ ]:
# Cell 1: fetch pipeline scripts (pinned to the PR branch) and download/verify the dataset
!wget -q -O train_ladd_tf.py https://raw.githubusercontent.com/mdabdullah003333-crypto/multimodel-food-quality-detection-dataset-litch-/vibe/ladd-thesis-figures-3afbd1/kaggle/train_ladd_tf.py
!wget -q -O make_thesis_figures.py https://raw.githubusercontent.com/mdabdullah003333-crypto/multimodel-food-quality-detection-dataset-litch-/vibe/ladd-thesis-figures-3afbd1/kaggle/make_thesis_figures.py
print('scripts fetched')
import urllib.request, zipfile, os, json, hashlib, shutil

BASE = '/kaggle/working/ladd'
WORK = '/kaggle/working'
os.makedirs(BASE, exist_ok=True)
H = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36'}

files = json.loads(urllib.request.urlopen(urllib.request.Request(
    'https://data.mendeley.com/public-api/datasets/6n9zbrp75r/files?version=1&folder_id=root',
    headers=H), timeout=60).read())

for f in files:
    name = f['filename']
    dest = os.path.join(WORK, name.replace(' ', '_'))
    print('downloading', name, '...')
    data = urllib.request.urlopen(urllib.request.Request(f['content_details']['download_url'], headers=H), timeout=1800).read()
    ok = hashlib.sha256(data).hexdigest() == f['content_details']['sha256_hash']
    print('  size', len(data), 'sha256 match:', ok)
    assert ok, 'checksum mismatch - download corrupted'
    open(dest, 'wb').write(data)
    if name.endswith('.zip') and 'Images' in name:  # Original_Images.zip -> images/
        with zipfile.ZipFile(dest) as z: z.extractall(BASE)

csv_src = os.path.join(WORK, 'Litchi_Sensor_Data.csv')
shutil.move(csv_src, os.path.join(BASE, 'Litchi_Sensor_Data.csv'))
for c in ['Fresh', 'Rotten', 'Formalin']: print(c, len(os.listdir(os.path.join(BASE, 'images', c))))
assert os.path.exists(os.path.join(BASE, 'Litchi_Sensor_Data.csv'))
print('DATASET_READY:', BASE)


In [ ]:
# Cell 2: train + evaluate + quantize (log tee'd for the figure script)
!sed -i 's|/kaggle/input/ladd-dataset|/kaggle/working/ladd|' train_ladd_tf.py
!python train_ladd_tf.py 2>&1 | tee /kaggle/working/tf_train_log.txt


In [ ]:
# Cell 3: regenerate all 6 thesis figures into /kaggle/working/thesis_figures/
!python make_thesis_figures.py


## Collect your outputs
Everything you need lands in `/kaggle/working/` (right-side **Output** panel -> *Download All*):
- `hybrid_freshness.keras` (float32 model), `hybrid_freshness_int8.tflite` (ESP32-S3 deployable)
- `tf_train_log.txt`, `fv_cache.npy`
- `thesis_figures/` — 6 figures x PNG + PDF (300 DPI)

**Expected (seed 42):** overall freshness MAE ~3.5/100 • Chemical-Masking recall ~96% (Formalin) • 0 false alerts on Fresh/Rotten • int8 |diff| mean ~0.17. Small (±0.1–0.3) drift vs the sandbox CPU run is normal GPU rounding.
